### Generate Data

In [1]:
import numpy as np
import math

L, d_k, d_v = 4, 8, 8
q = np.random.randn(L, d_k)
k = np.random.randn(L, d_k)
v = np.random.randn(L, d_v)

In [2]:
print("Q\n", q)
print("K\n", k)
print("V\n", v)

Q
 [[ 1.53731184  0.09576829 -2.3817861  -1.29182581  0.10158854 -1.86501173
  -1.76178612 -0.39185885]
 [ 0.61083824  0.24725296 -0.37533602  1.38475543  0.63731237 -0.31918449
  -0.40061496  1.21602559]
 [ 0.67928902  1.375495   -1.56175776  0.20957902  0.43045281  0.04091672
   0.26948395 -1.04669235]
 [ 0.49266447 -0.67607043  2.22702195  0.38964881  2.07918251  0.92552314
   0.14309359  0.21930194]]
K
 [[ 0.45118312 -1.16599978  0.47064875 -0.26831514  0.23271404  1.04879674
  -0.47066573 -0.18314242]
 [-0.43380504 -0.83966464 -0.88833852 -1.44414826 -0.39224294  1.43127663
  -0.13213359  0.03182626]
 [ 1.0487305  -1.00082554 -0.92419751 -0.56265853 -1.56619612 -0.43698081
  -0.16615819 -0.62673095]
 [ 0.06240445 -0.77310021 -0.13576373  0.46284496 -0.19344989  0.88146294
   0.73596564 -0.82721227]]
V
 [[ 1.25499536  0.98778345 -0.15904126 -1.0910635  -0.58597584  0.34392806
  -0.78665067  0.0895858 ]
 [ 1.32774952  0.74517162 -0.42166636  1.26686995  1.06236388 -1.03517124
   0.3

## Self Attention

$$
\text{self attention} = softmax\bigg(\frac{Q.K^T}{\sqrt{d_k}}+M\bigg)
$$

$$
\text{new V} = \text{self attention}.V
$$ 

In [3]:
np.matmul(q, k.T)

array([[-1.22382369,  0.74523809,  5.63866807, -2.88871291],
       [-0.78149751, -2.75414733, -1.59334629, -1.16653563],
       [-1.88067304, -0.54412624,  0.58037597,  0.30499105],
       [ 3.30120417, -1.68990085, -4.90620947,  0.76891228]])

In [4]:
# Why we need sqrt(d_k) in denominator
q.var(), k.var(), np.matmul(q, k.T).var()

(np.float64(1.1882189730610437),
 np.float64(0.5348511530914699),
 np.float64(5.8047871481411235))

In [5]:
scaled = np.matmul(q, k.T) / math.sqrt(d_k)
q.var(), k.var(), scaled.var()

(np.float64(1.1882189730610437),
 np.float64(0.5348511530914699),
 np.float64(0.7255983935176402))

Notice the reduction in variance of the product

In [6]:
scaled

array([[-0.43268701,  0.26348145,  1.99357021, -1.02131424],
       [-0.2763011 , -0.97373813, -0.56333298, -0.41243263],
       [-0.66491833, -0.19237768,  0.20519389,  0.10783062],
       [ 1.16715193, -0.59747018, -1.73460699,  0.27185154]])

## Masking

- This is to ensure words don't get context from words generated in the future. 
- Not required in the encoders, but required int he decoders

In [7]:
mask = np.tril(np.ones( (L, L) ))
mask

array([[1., 0., 0., 0.],
       [1., 1., 0., 0.],
       [1., 1., 1., 0.],
       [1., 1., 1., 1.]])

In [8]:
mask[mask == 0] = -np.inf
mask[mask == 1] = 0

In [9]:
mask

array([[  0., -inf, -inf, -inf],
       [  0.,   0., -inf, -inf],
       [  0.,   0.,   0., -inf],
       [  0.,   0.,   0.,   0.]])

In [10]:
scaled+mask

array([[-0.43268701,        -inf,        -inf,        -inf],
       [-0.2763011 , -0.97373813,        -inf,        -inf],
       [-0.66491833, -0.19237768,  0.20519389,        -inf],
       [ 1.16715193, -0.59747018, -1.73460699,  0.27185154]])

## Softmax

$$
\text{softmax} = \frac{e^{x_i}}{\sum_j e^x_j}
$$

In [11]:
def softmax(x):
  return (np.exp(x).T / np.sum(np.exp(x), axis=-1)).T

In [12]:
attention = softmax(scaled + mask)

In [13]:
attention

array([[1.        , 0.        , 0.        , 0.        ],
       [0.66761928, 0.33238072, 0.        , 0.        ],
       [0.20035089, 0.32137573, 0.47827338, 0.        ],
       [0.61174693, 0.10476257, 0.03360113, 0.24988937]])

In [14]:
new_v = np.matmul(attention, v)
new_v

array([[ 1.25499536e+00,  9.87783448e-01, -1.59041264e-01,
        -1.09106350e+00, -5.85975841e-01,  3.43928064e-01,
        -7.86650674e-01,  8.95857989e-02],
       [ 1.27917744e+00,  9.07143955e-01, -2.46332781e-01,
        -3.07331889e-01, -3.80995037e-02, -1.14457951e-01,
        -4.02573623e-01, -2.18462737e-01],
       [ 8.64890030e-01, -9.62979407e-03, -1.05829532e+00,
        -3.48734053e-01,  2.09377057e-01,  8.87646003e-01,
        -1.51798441e-01, -1.34961152e-01],
       [ 1.09612409e+00,  9.33359965e-01, -3.46433260e-01,
        -2.99892802e-01, -3.36021895e-01, -1.13133961e-01,
         1.19192334e-03, -5.32533800e-01]])

In [15]:
v

array([[ 1.25499536,  0.98778345, -0.15904126, -1.0910635 , -0.58597584,
         0.34392806, -0.78665067,  0.0895858 ],
       [ 1.32774952,  0.74517162, -0.42166636,  1.26686995,  1.06236388,
        -1.03517124,  0.36888282, -0.83720845],
       [ 0.39045477, -0.93463942, -1.86277965, -1.12337375, -0.03061038,
         2.40744868, -0.23572766,  0.24285008],
       [ 0.7049774 ,  1.13020143, -0.56974772,  1.09083478, -0.35143714,
        -1.18443093,  1.80759454, -2.03205749]])

## Function

In [16]:
def softmax(x):
  return (np.exp(x).T / np.sum(np.exp(x), axis=-1)).T

def scaled_dot_product_attention(q, k, v, mask=None):
  d_k = q.shape[-1]
  scaled = np.matmul(q, k.T) / math.sqrt(d_k)
  if mask is not None:
    scaled = scaled + mask
  attention = softmax(scaled)
  out = np.matmul(attention, v)
  return out, attention

In [17]:
values, attention = scaled_dot_product_attention(q, k, v, mask=mask)
print("Q\n", q)
print("K\n", k)
print("V\n", v)
print("New V\n", values)
print("Attention\n", attention)

Q
 [[ 1.53731184  0.09576829 -2.3817861  -1.29182581  0.10158854 -1.86501173
  -1.76178612 -0.39185885]
 [ 0.61083824  0.24725296 -0.37533602  1.38475543  0.63731237 -0.31918449
  -0.40061496  1.21602559]
 [ 0.67928902  1.375495   -1.56175776  0.20957902  0.43045281  0.04091672
   0.26948395 -1.04669235]
 [ 0.49266447 -0.67607043  2.22702195  0.38964881  2.07918251  0.92552314
   0.14309359  0.21930194]]
K
 [[ 0.45118312 -1.16599978  0.47064875 -0.26831514  0.23271404  1.04879674
  -0.47066573 -0.18314242]
 [-0.43380504 -0.83966464 -0.88833852 -1.44414826 -0.39224294  1.43127663
  -0.13213359  0.03182626]
 [ 1.0487305  -1.00082554 -0.92419751 -0.56265853 -1.56619612 -0.43698081
  -0.16615819 -0.62673095]
 [ 0.06240445 -0.77310021 -0.13576373  0.46284496 -0.19344989  0.88146294
   0.73596564 -0.82721227]]
V
 [[ 1.25499536  0.98778345 -0.15904126 -1.0910635  -0.58597584  0.34392806
  -0.78665067  0.0895858 ]
 [ 1.32774952  0.74517162 -0.42166636  1.26686995  1.06236388 -1.03517124
   0.3